In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torchinfo import summary

In [2]:
class Tokenizer:
    def __init__(self, dataset):
        self.vocab = set(dataset)
        self.word2idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

    def encode(self, text):
        return [self.word2idx[word] for word in text]

    def decode(self, indices):
        return ''.join([self.idx2word[idx] for idx in indices])


with open("../data/tinyshakespeare.txt", "r") as f:
    dataset = f.read()

    tokenizer = Tokenizer(dataset)
    train_data = torch.tensor(tokenizer.encode(dataset[:int(len(dataset) * 0.8)]), dtype=torch.long)
    test_data = torch.tensor(tokenizer.encode(dataset[int(len(dataset) * 0.8):]), dtype=torch.long)

    # reshape into (num_samples, block_size)
    block_size = 128
    train_data = train_data[:len(train_data) - (len(train_data) % block_size)].view(-1, block_size)
    test_data = test_data[:len(test_data) - (len(test_data) % block_size)].view(-1, block_size)

def batches(data, batch_size: int):
    x = torch.zeros((batch_size, block_size), dtype=torch.long)
    y = torch.zeros((batch_size, block_size), dtype=torch.long)
    for i in range(0, len(data) - batch_size, batch_size):
        x = data[i:i+batch_size]
        y = data[i+1:i+1+batch_size]
        yield x, y

vocab_size = len(tokenizer.vocab)
print(f"Vocab size: {vocab_size}")
print(f"Train data size: {train_data.shape}")
print(f"Test data size: {test_data.shape}")
print(f"Example encoded text:\n{tokenizer.decode(train_data[0].tolist())}")

Vocab size: 65
Train data size: torch.Size([6971, 128])
Test data size: torch.Size([1742, 128])
Example encoded text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to 


In [7]:
class RotaryPositionalEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) - applies rotation matrices to Q and K"""
    
    def __init__(self, head_dim, max_seq_length=2048, base=10000):
        super().__init__()
        self.head_dim = head_dim
        self.base = base
        
        # Precompute frequency inverse values
        inv_freq = 1.0 / (self.base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_length = max_seq_length
    
    def forward(self, x, seq_length):
        """
        Apply RoPE to queries or keys
        x: (batch_size, num_heads, seq_length, head_dim)
        """
        # Generate position indices
        t = torch.arange(seq_length, device=x.device).type_as(self.inv_freq)
        
        # Compute frequencies: (seq_length, head_dim // 2)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        
        # Duplicate frequencies for all dimensions: (seq_length, head_dim)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        # Compute cos and sin
        cos_cached = emb.cos()[None, None, :, :]  # (1, 1, seq_length, head_dim)
        sin_cached = emb.sin()[None, None, :, :]  # (1, 1, seq_length, head_dim)
        
        return cos_cached, sin_cached


def apply_rope(q, k, rope):
    """
    Apply rotary embeddings to queries and keys
    q, k: (batch_size, num_heads, seq_length, head_dim)
    """
    seq_length = q.shape[2]
    cos, sin = rope(q, seq_length)
    
    # Split into pairs for rotation
    q_rot = rotate_half(q, sin, cos)
    k_rot = rotate_half(k, sin, cos)
    
    return q_rot, k_rot


def rotate_half(x, sin, cos):
    """Rotate every two consecutive dimensions"""
    # x: (batch_size, num_heads, seq_length, head_dim)
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    
    rotated = torch.cat([
        x1 * cos[..., :x.shape[-1] // 2] - x2 * sin[..., x.shape[-1] // 2:],
        x1 * sin[..., :x.shape[-1] // 2] + x2 * cos[..., x.shape[-1] // 2:]
    ], dim=-1)
    
    return rotated


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads, max_seq_length=2048):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads
        assert self.head_dim * num_heads == embed_size, "Embed size must be divisible by num heads"
        
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)
        
        # Add RoPE
        self.rope = RotaryPositionalEmbedding(self.head_dim, max_seq_length)
    
    def forward(self, x):
        batch_size, seq_length, _ = x.size()
        
        # Linear projections
        Q = self.query(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply RoPE to Q and K
        Q, K = apply_rope(Q, K, self.rope)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attention_weights, V).transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_size)
        out = self.fc_out(out)
        
        return out

class Block(nn.Module):
    def __init__(self, embed_size, num_heads, block_num):
        super(Block, self).__init__()
        if block_num % 2 == 0:
            self.attention = MultiHeadAttention(embed_size, num_heads)
        else:
            self.attention = nn.Identity()

        self.ffn = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.norm1(self.attention(x))
        x = x + self.norm2(self.ffn(x))
        return x
    
class LLM(nn.Module):
    def __init__(self, vocab_size, embed_size, num_blocks, num_heads=4):
        super(LLM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.blocks = nn.ModuleList([Block(embed_size, num_heads, i) for i in range(num_blocks)])
        self.output_layer = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks: 
            x = block(x)
        output = self.output_layer(x)
        return output

model = LLM(vocab_size, embed_size=64, num_blocks=8)
summary(model, input_size=(1, 10), dtypes=[torch.long])

Layer (type:depth-idx)                                  Output Shape              Param #
LLM                                                     [1, 10, 65]               --
├─Embedding: 1-1                                        [1, 10, 64]               4,160
├─ModuleList: 1-2                                       --                        --
│    └─Block: 2-1                                       [1, 10, 64]               --
│    │    └─MultiHeadAttention: 3-1                     [1, 10, 64]               16,640
│    │    └─LayerNorm: 3-2                              [1, 10, 64]               128
│    │    └─Sequential: 3-3                             [1, 10, 64]               33,088
│    │    └─LayerNorm: 3-4                              [1, 10, 64]               128
│    └─Block: 2-2                                       [1, 10, 64]               --
│    │    └─Identity: 3-5                               [1, 10, 64]               --
│    │    └─LayerNorm: 3-6                     

In [8]:
import time

@torch.inference_mode()
def generate(model, tokenizer, prompt, max_length=20, temperature=1.0, verbose=True):

    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0)

    start_time = time.time()
    for _ in range(max_length - len(input_ids[0])):
        output = model(input_ids)
        next_token_logits = output[:, -1, :] / temperature
        next_token = torch.multinomial(F.softmax(next_token_logits, dim=-1), num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)
    end_time = time.time()

    output = tokenizer.decode(input_ids.squeeze().tolist())

    if verbose:
        print(f"Total Time: {end_time - start_time:.4f} seconds")
        print(f"Tokens per second: {input_ids.size(1) / (end_time - start_time):.2f}")

    return output


print(generate(model, tokenizer, max_length=512, prompt="MIRANDA:"))

Total Time: 1.3947 seconds
Tokens per second: 367.09
MIRANDA:Pff$wff$PFJQYI',zI
I
II'xI
I
I
I
i$
IGvI
I
iQ
vI
iv
iI
I
I
GiqPqiIiIGIGQQqN
GNi.I
GuaaEiQqi,&
I
iQamuaEIGvI&I&
iuauaEiuaEi.Iiuua:iqIG:iuaEifqiuua:xI
iqiuuauu.IGI&
i.I
i.iQqi.I
;uaUqI
IG:IGIGI
IG:S&
iIiaEIiuuuauqIiuauaEiuaEIGvI&
iuuua
ifuaEIGQqiuuqiuqI
i.IbaEiuuauuuauuauauqIiuauqIiuaUqi.IGIGI&
i.I
iI
;iuaEfqIG:i.uaEfuqi.uaEI&
I&
iuqIGIiYIiqiuauaEIiQauaEIGi.iQqIQqiDauqIGuqiuaEi.Ii.IaEI
IGIaEIGvIi.IiuqiuauuuauuMxI&
i.IiuaEfI&
iuaUqi.IGuaEIi$.IGvuauaEi.IGuqiuaEiua;uuauqIauqiIGIi.I
uqIifuqi.I&;&
Iiuauuqi


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=1000)

In [13]:
for epoch in range(10):
    model.train()
    pbar = tqdm(batches(train_data, batch_size=128), total=len(train_data), desc=f"Epoch {epoch+1}")
    for x, y in pbar:
        output = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        pbar.set_postfix_str(f"loss: {loss.item():.4f}, lr: {scheduler.get_last_lr()[0]:.1e}")

    print(generate(model, tokenizer, max_length=128, prompt="MIRANDA:"))

Epoch 1:   1%|          | 54/6971 [00:08<17:27,  6.60it/s, loss: 3.3483, lr: 3.7e-04]


Total Time: 0.1358 seconds
Tokens per second: 942.58
MIRANDA:SenattSae.  edrao hesuHhyYlde ohLe ydahhh
oTtpdl podme r;Ias.a:gy nt 
lso he hith nyodr oov- ehn
dtat. o  e lon !yc    a


Epoch 2:   1%|          | 54/6971 [00:07<16:38,  6.92it/s, loss: 3.3475, lr: 3.5e-04]


Total Time: 0.1580 seconds
Tokens per second: 809.96
MIRANDA:l'
Na,
rti
etli re
eoNrinlo!S epYeYoyssohg hte eIenW
erhGtEGrenaooie rs ttr yrs
tho
dwrh nCeutNrytautav'od ealrdq,utaloe


Epoch 3:   1%|          | 48/6971 [00:07<17:00,  6.78it/s, loss: 3.2450, lr: 3.3e-04]


KeyboardInterrupt: 

In [14]:
print(generate(model, tokenizer, max_length=128, prompt="MIRANDA:"))

Total Time: 0.1594 seconds
Tokens per second: 803.21
MIRANDA:.s'vrm wlit 
oub lde

n tf a m e
aiK areet
aststttCLyenboo. hretu
tkNu.hsU
meuwdPt
pLuroyo ,s a y  seAd b lL
 e o oIpYo:


In [15]:
gradients = []
for name, param in model.named_parameters():
    if param.grad is not None:
        weight_norm = param.data.norm().item()
        grad_norm = param.grad.norm().item()
        ratio = grad_norm / (param.data.norm().item() + 1e-8)  # Avoid division by zero
        gradients.append({"name": name, "weight_norm": weight_norm, "grad_norm": grad_norm, "ratio": ratio})

# Show all
show_weights = True
show_biases = False

print(f"{"Name":<45} {"Weight Norm":<25} {"Grad Norm":<25} {"Ratio":<25}")
for grad in gradients:
    if (show_weights and "weight" in grad['name']) or (show_biases and "bias" in grad['name']):
        print(f"{grad['name']:<45} {grad['weight_norm']:<25.4e} {grad['grad_norm']:<25.4e} {grad['ratio']:<25.4e}")

Name                                          Weight Norm               Grad Norm                 Ratio                    
embedding.weight                              6.2505e+01                1.0458e-02                1.6732e-04               
blocks.0.attention.query.weight               4.6548e+00                1.2507e-02                2.6868e-03               
blocks.0.attention.key.weight                 4.6784e+00                7.7498e-03                1.6565e-03               
blocks.0.attention.value.weight               4.6234e+00                9.8620e-02                2.1331e-02               
blocks.0.attention.fc_out.weight              4.6143e+00                1.5550e-01                3.3698e-02               
blocks.0.ffn.0.weight                         9.5456e+00                9.2929e-02                9.7353e-03               
blocks.0.ffn.2.weight                         4.9527e+00                2.3029e-01                4.6497e-02               
blocks.0